<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_14_file_io_json/note_lesson_14_file_io_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Файли I/O, менеджери контексту, JSON

> Сценарій: менеджер ресторану надсилає `orders.csv` — список замовлень за тиждень. Завдання — прочитати файл, порахувати аналітику і зберегти результат у `report.json`, який потім читає frontend (сайт чи мобільний застосунок).

У Уроках 5–6 ми вже рахували аналітику ресторану — середні чеки, лідера дня, чайові по офіціантах — але тільки в оперативній пам'яті: закриваєш ноутбук, і весь `report` зникає. Сьогодні той самий тип звіту (`report.json`) переживає завершення програми — це і є відсутня частина: не нова аналітика, а її збереження.

## 🔁 RETRIEVE — пригадай попередні уроки (без підглядання)

1. Що поверне `{"name": "Alice"}["age"]` — і як цього уникнути через `.get()`?
2. Напиши f-string, який виводить `"Ціна: 320.00 грн"` зі змінних `price = 320` і `currency = "грн"`.
3. Чим list comprehension `{d["city"] for d in orders}` відрізняється від звичайного списку — яка структура даних вийде?

Це не нова тема — `.get()` (Урок 6) і f-strings (Урок 3) сьогодні просто застосовуються в новому контексті: читанні й записі файлів.

## 📖 CONCEPT

### 1. Ментальна модель: RAM проти диска

Коли Python-програма завершується, усі її змінні зникають — вони жили в **RAM** (оперативній пам'яті), тимчасово. Файли на диску (`orders.csv`, `report.json`) — постійні: вони залишаються між запусками програми.

```
RAM (оперативна пам'ять)              DISK (диск)
scores = {"Alice": 150}  → зникає     orders.csv   → залишається
df = DataFrame(...)      → зникає     report.json  → залишається
```

Менеджер ресторану вже зберіг дані у файл `orders.csv` на диску. План на цей урок:
**прочитати** файл (диск → RAM) → обробити дані (у RAM) → **записати** результат (RAM → диск).

### 2. Що таке файл у Python

Python не звертається до диска напряму — він просить **операційну систему (OS)** відкрити з'єднання до файлу. OS повертає **file object** (file handle) — об'єкт-посилання, через який Python читає або пише дані.

```python
file_object = open("orders.csv", "r")
#              ім'я файлу      режим (r = read)
```

**Режими відкриття:**

| Режим | Що робить | Ризик |
|---|---|---|
| `r` | Читання | `FileNotFoundError`, якщо файл відсутній |
| `w` | Запис | **Стирає** весь наявний вміст; якщо файлу нема — створює |
| `a` | Дописування в кінець | Якщо файлу нема — створює |

Завжди вказуй `encoding="utf-8"` — інакше кирилиця та інші не-ASCII символи можуть прочитатись некоректно.

### 3. `with open(...)` — навіщо

Ручне закриття файлу небезпечне: якщо між `open()` і `f.close()` станеться помилка, `close()` ніколи не виконається — файл лишиться заблокованим (*resource leak*).

```python
# НЕБЕЗПЕЧНО:
f = open("orders.csv", "r")
data = f.read()
result = 10 / 0     # виняток — f.close() нижче вже не виконається
f.close()
```

`with` — контекстний менеджер: закриває файл автоматично, навіть якщо всередині блоку сталася помилка.

```python
with open("orders.csv", "r", encoding="utf-8") as f:
    data = f.read()
# файл закрито тут — гарантовано
```

In [1]:
# Читаємо звичайний текстовий файл
with open("restaurant_info.txt", "r", encoding="utf-8") as f:
    text = f.read()   # весь вміст файлу — одним рядком

print(text)
print(f"Тип даних: {type(text)}")

Restaurant Demo Dataset
Навчальний набір даних для уроку Python: File I/O, JSON, CSV, datetime.

Сценарій:
  Менеджер ресторану надсилає CSV-файл з замовленнями за тиждень.
  Наше завдання:
    1. Прочитати CSV-файл
    2. Перетворити текстову дату у справжній datetime
    3. Порахувати базову аналітику
    4. Зберегти результат у JSON для фронтенду

Файли:
  orders.csv          — вхідні дані (замовлення)
  config.json         — конфігурація пайплайну
  report.json         — вихідний файл (генерується Python-ом)

Тип даних: <class 'str'>


`f.read()` повертає **один рядок** (`str`) — переноси рядків `\n` теж входять у нього. Для великого файлу (наприклад, 50 ГБ) `f.read()` завантажить усе в RAM і програма впаде. Альтернатива — читати рядок за рядком, не тримаючи весь файл у пам'яті одночасно: `for line in f: ...`.

### 4. Чому `str()` не годиться для збереження словника

In [2]:
my_data = {"name": "Alice", "score": 150, "active": True}

with open("bad_data.txt", "w", encoding="utf-8") as f:
    f.write(str(my_data))

with open("bad_data.txt", "r", encoding="utf-8") as f:
    loaded = f.read()

print(f"Прочитано: {loaded!r}")
print(f"Тип: {type(loaded)}")

Прочитано: "{'name': 'Alice', 'score': 150, 'active': True}"
Тип: <class 'str'>


`loaded` — це `str`, не словник. Три проблеми: одинарні лапки (Python-синтаксис, не JSON), `True` з великої літери (JSON очікує `true`), і головне — **тип даних втрачено**: щоб знову отримати словник, довелося б писати власний парсер.

### 5. JSON вирішує цю проблему

JSON (JavaScript Object Notation) — текстовий, але **структурований** і **типізований** формат, який розуміє практично будь-яка мова програмування.

| Python | JSON |
|---|---|
| `dict` | `{}` (object) |
| `list` | `[]` (array) |
| `str` | `""` (подвійні лапки) |
| `int` / `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

In [3]:
import json

my_data = {"name": "Alice", "score": 150, "active": True}

json_string = json.dumps(my_data, indent=2)   # dict → JSON-рядок

print("Python dict:", my_data)
print()
print("JSON-рядок (json.dumps):")
print(json_string)

Python dict: {'name': 'Alice', 'score': 150, 'active': True}

JSON-рядок (json.dumps):
{
  "name": "Alice",
  "score": 150,
  "active": true
}


In [4]:
# Повний цикл: dict → JSON-рядок → dict
serialized = json.dumps(my_data)     # серіалізація
restored = json.loads(serialized)    # десеріалізація

print(f"Оригінал:   {my_data}")
print(f"JSON-рядок: {serialized}")
print(f"Відновлено: {restored}")
print(f"Тип 'active' після відновлення: {type(restored['active']).__name__}")

Оригінал:   {'name': 'Alice', 'score': 150, 'active': True}
JSON-рядок: {"name": "Alice", "score": 150, "active": true}
Відновлено: {'name': 'Alice', 'score': 150, 'active': True}
Тип 'active' після відновлення: bool


### 6. `pipeline_config.json` — налаштування замість хардкоду

```python
# Хардкод — погано: щоб змінити файл, треба лізти в код
df = pd.read_csv("orders.csv")

# Через конфіг — добре: щоб змінити файл, достатньо відредагувати pipeline_config.json
config = json.load(config_file)
df = pd.read_csv(config["input_file"])
```

`json.load(file)` читає JSON **з файлу**, `json.loads(string)` — **з рядка**. Суфікс `s` = *string*.

In [5]:
with open("pipeline_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

print("Конфіг завантажено:")
for key, value in config.items():
    print(f"  config[{key!r}] = {value!r}")

Конфіг завантажено:
  config['project_name'] = 'Restaurant Weekly Report'
  config['currency'] = 'UAH'
  config['input_file'] = 'orders.csv'
  config['output_file'] = 'report.json'
  config['date_column'] = 'order_date'


### 7. `orders.csv` через pandas

CSV (Comma-Separated Values) — теж текстовий файл, але зі структурою таблиці: перший рядок — заголовки колонок, кожен наступний — один запис.

Читати CSV вручну через `open()` можливо, але незручно (треба самому розбивати кожен рядок по комі й визначати типи). `pandas.read_csv()` робить це автоматично: читає файл, розпізнає заголовки, повертає `DataFrame` із розпізнаними типами колонок.

In [6]:
import pandas as pd

df = pd.read_csv(config["input_file"])   # беремо шлях із конфігу, не хардкодимо

print(f"DataFrame: {df.shape[0]} рядків, {df.shape[1]} колонок")
df.head()

DataFrame: 12 рядків, 6 колонок


,order_id,customer_name,dish,price,order_date,city
0,1,Anna,Pizza,320,2026-03-10 12:30,Kyiv
1,2,Oleh,Burger,210,2026-03-10 13:10,Lviv
2,3,Ira,Pasta,280,2026-03-11 18:45,Kyiv
3,4,Max,Soup,150,2026-03-11 14:20,Odesa
4,5,Nina,Pizza,320,2026-03-12 19:05,Lviv


In [7]:
print("Типи колонок:")
print(df.dtypes)
print()
print(f"order_date має тип {df['order_date'].dtype} — для Python це просто рядок, не дата.")

Типи колонок:
order_id         int64
customer_name      str
dish               str
price            int64
order_date         str
city               str
dtype: object

order_date має тип str — для Python це просто рядок, не дата.


**PREDICT.** Поки `order_date` — рядок (`object`), спробувати `df["order_date"].dt.day_name()` дасть помилку. Яку саме — `AttributeError` чи `TypeError`? Зафіксуй відповідь подумки, тоді запусти наступну клітинку.

`pd.to_datetime()` приймає колонку рядків і перетворює кожне значення на справжній `datetime64[ns]` — після цього стають доступні `.dt.day_name()`, `.dt.hour`, `.dt.month` тощо.

In [8]:
print("До перетворення:", repr(df["order_date"].iloc[0]), type(df["order_date"].iloc[0]).__name__)

df[config["date_column"]] = pd.to_datetime(df[config["date_column"]])

print("Після перетворення:", df["order_date"].iloc[0], "| dtype:", df["order_date"].dtype)

До перетворення: '2026-03-10 12:30' str
Після перетворення: 2026-03-10 12:30:00 | dtype: datetime64[us]


In [9]:
df["day_name"] = df["order_date"].dt.day_name()
df["hour"] = df["order_date"].dt.hour

print(df[["order_date", "day_name", "hour", "dish", "price", "city"]].to_string(index=False))

         order_date  day_name  hour   dish  price  city
2026-03-10 12:30:00   Tuesday    12  Pizza    320  Kyiv
2026-03-10 13:10:00   Tuesday    13 Burger    210  Lviv
2026-03-11 18:45:00 Wednesday    18  Pasta    280  Kyiv
2026-03-11 14:20:00 Wednesday    14   Soup    150 Odesa
2026-03-12 19:05:00  Thursday    19  Pizza    320  Lviv
2026-03-12 11:50:00  Thursday    11  Salad    190  Kyiv
2026-03-13 16:40:00    Friday    16 Burger    210 Odesa
2026-03-13 20:15:00    Friday    20  Pasta    280  Kyiv
2026-03-14 13:00:00  Saturday    13  Pizza    320  Lviv
2026-03-14 19:30:00  Saturday    19   Soup    150  Kyiv
2026-03-15 12:10:00    Sunday    12  Salad    190 Odesa
2026-03-15 20:45:00    Sunday    20  Pasta    280  Kyiv


### 8. Базова аналітика

`pandas` повертає суми/середні як `numpy.int64` / `numpy.float64`, не звичайний Python `int`/`float`. `json.dumps()` не вміє серіалізувати numpy-типи — тому їх явно обгортають у `float()`/`int()` перед тим, як класти в результат (детальніше — нижче, у розділі про типову помилку).

In [10]:
total_orders = len(df)
total_revenue = float(df["price"].sum())
average_price = float(df["price"].mean())

print(f"Замовлень:        {total_orders}")
print(f"Виручка:          {total_revenue:,.0f} {config['currency']}")
print(f"Середня ціна:     {average_price:.2f} {config['currency']}")

Замовлень:        12
Виручка:          2,900 UAH
Середня ціна:     241.67 UAH


In [11]:
revenue_by_city = {city: float(v) for city, v in df.groupby("city")["price"].sum().items()}
print("Виручка по містах:", revenue_by_city)

Виручка по містах: {'Kyiv': 1500.0, 'Lviv': 850.0, 'Odesa': 550.0}


In [12]:
top_dishes = {dish: int(count) for dish, count in df["dish"].value_counts().head(3).items()}
print("Топ страви:")
for dish, count in top_dishes.items():
    print(f"  {dish:<10} — {count} замовлень")

Топ страви:
  Pizza      — 3 замовлень
  Pasta      — 3 замовлень
  Burger     — 2 замовлень


### Типова помилка: numpy-тип у `json.dumps()`

`df["price"].sum()` без обгортки в `float()` повертає `numpy.float64` — технічно схоже на звичайне число, але `json` про нього не знає нічого. Побачимо це наочно:

In [13]:
import numpy as np

raw_sum = df["price"].sum()   # БЕЗ float() — це numpy.float64
print(f"Тип raw_sum: {type(raw_sum)}")

try:
    json.dumps({"total": raw_sum})
except TypeError as e:
    print(f"TypeError: {e}")

# Виправлення — обгорнути у звичайний Python float
fixed = json.dumps({"total": float(raw_sum)})
print(f"Після float(): {fixed}")

Тип raw_sum: <class 'numpy.int64'>
TypeError: Object of type int64 is not JSON serializable
Після float(): {"total": 2900.0}


### 9. Формуємо звіт для frontend

In [14]:
report = {
    "project": config["project_name"],
    "currency": config["currency"],
    "total_orders": total_orders,
    "total_revenue": total_revenue,
    "average_price": round(average_price, 2),
    "revenue_by_city": revenue_by_city,
    "top_dishes": top_dishes,
}

for key, value in report.items():
    print(f"  {key:<20}: {value}")

  project             : Restaurant Weekly Report
  currency            : UAH
  total_orders        : 12
  total_revenue       : 2900.0
  average_price       : 241.67
  revenue_by_city     : {'Kyiv': 1500.0, 'Lviv': 850.0, 'Odesa': 550.0}
  top_dishes          : {'Pizza': 3, 'Pasta': 3, 'Burger': 2}


### 10. Записуємо у `report.json`

`json.dump(obj, file)` пише Python-об'єкт **у файл**; `json.dumps(obj)` — у рядок (для логів, дебагу, відповіді API). `ensure_ascii=False` дозволяє UTF-8 (кирилиця не ескейпується), `indent=4` робить файл читабельним.

In [15]:
with open(config["output_file"], "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

print(f"Файл '{config['output_file']}' збережено.")

Файл 'report.json' збережено.


### 11. Перевіряємо: читаємо файл назад

Симулюємо те, що робив би frontend — читає `report.json` і показує дані.

In [16]:
with open(config["output_file"], "r", encoding="utf-8") as f:
    saved_report = json.load(f)

print(f"{saved_report['project']}")
print(f"Замовлень:    {saved_report['total_orders']}")
print(f"Виручка:      {saved_report['total_revenue']:,.0f} {saved_report['currency']}")
print(f"Середній чек: {saved_report['average_price']:.2f} {saved_report['currency']}")
print("По містах:")
for city, revenue in saved_report["revenue_by_city"].items():
    print(f"  {city:<8} {revenue:>8,.0f} {saved_report['currency']}")

Restaurant Weekly Report
Замовлень:    12
Виручка:      2,900 UAH
Середній чек: 241.67 UAH
По містах:
  Kyiv        1,500 UAH
  Lviv          850 UAH
  Odesa         550 UAH


### 12. Весь пайплайн в одному місці

Так виглядає реальний скрипт обробки даних — ті самі кроки, зібрані разом, без проміжних пояснень:

In [17]:
import json
import pandas as pd

with open("pipeline_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

df = pd.read_csv(config["input_file"])
df[config["date_column"]] = pd.to_datetime(df[config["date_column"]])

total_orders = len(df)
total_revenue = float(df["price"].sum())
average_price = float(df["price"].mean())
revenue_by_city = {c: float(v) for c, v in df.groupby("city")["price"].sum().items()}
top_dishes = {d: int(c) for d, c in df["dish"].value_counts().head(3).items()}

report = {
    "project": config["project_name"],
    "currency": config["currency"],
    "total_orders": total_orders,
    "total_revenue": total_revenue,
    "average_price": round(average_price, 2),
    "revenue_by_city": revenue_by_city,
    "top_dishes": top_dishes,
}

with open(config["output_file"], "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

print(f"Пайплайн завершено: {total_orders} замовлень → '{config['output_file']}'")

Пайплайн завершено: 12 замовлень → 'report.json'


## 🛠️ CREATE — самостійні завдання

Працюй із тим самим `df`/`config`/`report`, що вже є в пам'яті вище.

### Завдання 1 — `revenue_by_dish`

Порахуй виручку по стравах (аналогічно `revenue_by_city`) і додай ключ `"revenue_by_dish"` у `report`.

In [18]:
# TODO: порахуй revenue_by_dish так само, як revenue_by_city вище (groupby("dish"))
# BEGIN SOLUTION
revenue_by_dish = {dish: float(v) for dish, v in df.groupby("dish")["price"].sum().items()}
report["revenue_by_dish"] = revenue_by_dish
# END SOLUTION

print(revenue_by_dish)
assert "revenue_by_dish" in report
assert revenue_by_dish["Pizza"] == 960.0
print("OK")

{'Burger': 420.0, 'Pasta': 840.0, 'Pizza': 960.0, 'Salad': 380.0, 'Soup': 300.0}
OK


### Завдання 2 — `revenue_by_month`

Додай колонку `month` (підказка: `df["order_date"].dt.to_period("M")`, дає рядки виду `"2026-03"`) і порахуй виручку по місяцях.

In [19]:
# TODO: df["month"] = ... , потім groupby("month") аналогічно
# BEGIN SOLUTION
df["month"] = df["order_date"].dt.to_period("M").astype(str)
revenue_by_month = {m: float(v) for m, v in df.groupby("month")["price"].sum().items()}
report["revenue_by_month"] = revenue_by_month
# END SOLUTION

print(revenue_by_month)
assert "revenue_by_month" in report
assert revenue_by_month["2026-03"] == total_revenue
print("OK")

{'2026-03': 2900.0}
OK


### Завдання 3 — `report_extended.json`

Запиши `report` (тепер уже з `revenue_by_dish` і `revenue_by_month`) у новий файл `report_extended.json`.

In [20]:
# TODO: збережи report у "report_extended.json" тим самим способом, що й report.json вище
# BEGIN SOLUTION
with open("report_extended.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)
# END SOLUTION

with open("report_extended.json", "r", encoding="utf-8") as f:
    check = json.load(f)
assert "revenue_by_dish" in check and "revenue_by_month" in check
print("OK — report_extended.json збережено і перевірено.")

OK — report_extended.json збережено і перевірено.


### Завдання 4 — телефонна книга: завантаження з fallback

Звіт-пайплайн вище щоразу *перезаписує* файл заново. Але часто потрібен інший патерн: **дозавантажити** дані, які вже існують на диску, дописати щось нове і зберегти назад — так працює будь-який застосунок з "збереженим станом" (нотатки, контакти, кошик покупок).

Ключова відмінність від пайплайну вище: файл може ще **не існувати** при першому запуску — тоді `open(path, "r")` кине `FileNotFoundError`, і потрібно стартувати з порожньої структури, а не падати.

Напиши три функції за такою логікою:

- `load_phonebook(path)` — намагається прочитати JSON з `path`; якщо файл відсутній (`FileNotFoundError`) — повертає порожній `{}`.
- `add_contact(phonebook, first_name, last_name, phone, balance)` — додає запис за ключем `f"{first_name} {last_name}"` (рядковий ключ, бо JSON не дозволяє кортежі як ключі) зі значенням `{"phone": ..., "balance": ...}`; повертає оновлений `phonebook`.
- `save_phonebook(phonebook, path)` — записує `phonebook` у `path` через `json.dump(..., ensure_ascii=False, indent=4)`.


In [21]:
PHONEBOOK_FILE = "contacts.json"

# BEGIN SOLUTION
def load_phonebook(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {}


def add_contact(phonebook, first_name, last_name, phone, balance):
    key = f"{first_name} {last_name}"
    phonebook[key] = {"phone": phone, "balance": balance}
    return phonebook


def save_phonebook(phonebook, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(phonebook, f, ensure_ascii=False, indent=4)
# END SOLUTION

# перший запуск — файлу ще нема
phonebook = load_phonebook(PHONEBOOK_FILE)
print("Завантажено:", phonebook)

phonebook = add_contact(phonebook, "Alice", "Smith", "050-123-4567", 1500.5)
phonebook = add_contact(phonebook, "Bob", "Jones", "097-987-6543", 250.0)
save_phonebook(phonebook, PHONEBOOK_FILE)
print("Збережено:", phonebook)

# другий "запуск" — тепер файл уже є, fallback не спрацьовує
reloaded = load_phonebook(PHONEBOOK_FILE)
assert reloaded == phonebook
assert reloaded["Alice Smith"]["phone"] == "050-123-4567"
print("OK — телефонна книга завантажується і зберігається коректно.")


Завантажено: {'Alice Smith': {'phone': '050-123-4567', 'balance': 1500.5}, 'Bob Jones': {'phone': '097-987-6543', 'balance': 250.0}}
Збережено: {'Alice Smith': {'phone': '050-123-4567', 'balance': 1500.5}, 'Bob Jones': {'phone': '097-987-6543', 'balance': 250.0}}


OK — телефонна книга завантажується і зберігається коректно.


## Типові помилки

| Помилка | Причина | Виправлення |
|---|---|---|
| `FileNotFoundError` | Файл не існує в режимі `r` | Перевір шлях або обгорни у `try`/`except` |
| `TypeError: Object of type int64 is not JSON serializable` | numpy-тип замість Python | `float(val)` або `int(val)` — демонстрація вище |
| `AttributeError: Can only use .dt accessor...` | Колонка дати ще `str` | `pd.to_datetime(df[col])` перед `.dt.*` |
| `JSONDecodeError` при читанні | Файл записано через `str()`, не `json.dump()` | Завжди `json.dump()`/`json.dumps()` для запису |
| Файл порожній після краху програми | `open()` без `with` | `with open(...) as f:` — гарантоване закриття |

## Підсумок

1. **RAM проти диска.** Змінні — тимчасові (RAM), файли — постійні (диск). `open()` просить OS відкрити з'єднання до файлу.
2. **`with open(...) as f:`** — завжди. Гарантує закриття файлу навіть при помилці всередині блоку.
3. **Формати файлів:** `.txt` — просто текст без структури; `.json` — структурований, зберігає типи даних; `.csv` — таблиця, зручна для `pandas`.
4. **Дата в CSV — завжди текст.** Потрібен `pd.to_datetime()`, щоб отримати справжній `datetime` і `.dt.*`-атрибути.
5. **JSON — міст між системами.** `json.dump()` записує файл, який може прочитати будь-який frontend чи інший backend.

## ✅ Самоперевірка (5 запитань)

**1.** Чому `with open(...) as f:` безпечніший за ручний `f = open(...)` + `f.close()`?

<details><summary>Відповідь</summary>Якщо між <code>open()</code> і <code>close()</code> станеться помилка, ручний <code>f.close()</code> ніколи не виконається — файл лишиться заблокованим (resource leak). <code>with</code> закриває файл автоматично, навіть якщо всередині блоку виникло виключення.</details>

**2.** У чому різниця між збереженням словника через `f.write(str(my_dict))` і через `json.dump(my_dict, f)`?

<details><summary>Відповідь</summary><code>str()</code> дає рядок у Python-синтаксисі (одинарні лапки, <code>True</code> з великої літери) — тип даних втрачається, прочитати назад як словник напряму не можна. <code>json.dump()</code> зберігає структурований, типізований JSON, який <code>json.load()</code> коректно відновлює як <code>dict</code>.</details>

**3.** Чому `df["price"].sum()` без `float()` ламає `json.dumps()`?

<details><summary>Відповідь</summary><code>pandas</code>/<code>numpy</code> повертають <code>numpy.int64</code>/<code>numpy.float64</code> — це не вбудовані Python-типи, і стандартний модуль <code>json</code> не знає, як їх серіалізувати. <code>float()</code>/<code>int()</code> конвертують у звичайний Python-тип, який <code>json</code> розуміє.</details>

**4.** Навіщо винести назви файлів у `pipeline_config.json` замість того, щоб писати `pd.read_csv("orders.csv")` прямо в коді?

<details><summary>Відповідь</summary>Щоб змінити вхідний файл, достатньо відредагувати <code>pipeline_config.json</code>, не чіпаючи код. Так само працює конфігурація більшості реальних Python-програм і web-серверів.</details>

**5.** Чому `load_phonebook` ловить саме `FileNotFoundError`, а не просто повертає `{}` без `try`/`except`?

<details><summary>Відповідь</summary>Без <code>try</code>/<code>except</code> виклик <code>open(path, "r")</code> на неіснуючому файлі одразу кине <code>FileNotFoundError</code> і зупинить програму. <code>try</code>/<code>except</code> перетворює "файла ще нема" з аварії на очікуваний, оброблений сценарій — типовий патерн для першого запуску застосунку зі збереженим станом.</details>

## Далі

Наступна тема — Git + GitHub (Урок 15): перший репозиторій, pull request, портфоліо.